In [1]:
import numpy as np
from matplotlib import pyplot as plt


# Generating Random Points

generate "NUMBER_OF_POINTS" points in the chosen limit and save it in "DESTINATION_FILE"

In [2]:
DESTINATION_FILE = "10points.csv"
NUMBER_OF_POINTS = 10
X_RANGE = (0,100)
Y_RANGE = (0,100)

SEED = 42
rng = np.random.default_rng(SEED)

generated_points = rng.uniform(
    low=[X_RANGE[0], Y_RANGE[0]],
    high=[X_RANGE[1], Y_RANGE[1]],
    size=(NUMBER_OF_POINTS, 2)
)
generated_points = np.round(generated_points, 2)

np.savetxt(
    DESTINATION_FILE,
    generated_points,
    delimiter=",",
    header="x,y",
    comments="",
    fmt="%.2f"
)

# Load Data
reads the "DATA_FILE" which contains a header x, y and each row has two columns. saves the number of points into "n".

In [3]:
DATA_FILE = "10points.csv"

data = np.loadtxt("10points.csv", delimiter=",", skiprows=1)
n = data.shape[0]

In [4]:
data

array([[77.4 , 43.89],
       [85.86, 69.74],
       [ 9.42, 97.56],
       [76.11, 78.61],
       [12.81, 45.04],
       [37.08, 92.68],
       [64.39, 82.28],
       [44.34, 22.72],
       [55.46,  6.38],
       [82.76, 63.17]])

# Distance Matrix   
dist_matrix<sub>ij</sub> holds Euclidean distance between point<sub>i</sub> and point<sub>i</sub> 

In [5]:
diff = data[:, None, :] - data[None, :, :] # The distance between all pairs
dist_matrix = np.linalg.norm(diff, axis=2) # Root of sum of square of elements in the distance vector

# Initial Population
chooses "POPULATION_SIZE" random permutations 

In [6]:
POPULATION_SIZE = 100

SEED = 42
rng = np.random.default_rng(SEED)

indices = list(range(n))

# Random permutation of indices
population = np.array([
    rng.permutation(indices) for _ in range(POPULATION_SIZE)
])

# Fitness Function
uses inverse of Total distance to make minimization into maximization problem

In [7]:
def fitness_of(c):
    Total_distance = 0
    
    for i in range(1,n):
        prev_point = c[i-1]
        next_point = c[i]

        # Distance between two consecutive points in the chromosome
        Total_distance += dist_matrix[prev_point][next_point] 

    return 1 / Total_distance
    

# Crossover
Uses Ordered Crossover. Puts a random chunk of the first parent in the child and fills the left part with other parent by keeping the order 

In [8]:
def crossover(parent_a, parent_b, n, rng):

    # Empty array for child
    child = np.array([-1] * n)
    
    # Keeps used indices
    used = set() 

    # Place a random part of the first parent in child
    shared_range = rng.choice(n, size=2, replace=False)
    # Put the smaller one first
    shared_range.sort()

    for i in range(shared_range[0],shared_range[1]):
        child[i] = parent_a[i]
        used.add(parent_a[i])
    
    # Fill remaining part with the second parent keeping its order
    p_ptr = 0 # Parent pointer 
    for i in range(0,n):
        # If was not filled with parent_a 
        if child[i] == -1:

            # moves forward to find the first not used point in parent_b 
            while parent_b[p_ptr] in used:
                p_ptr += 1
            
            # Places the first unused point
            child[i] = parent_b[p_ptr]
            p_ptr += 1
            # point will not be seen again so it won't be necessary to add it into used set 

    return child

# Mutation
Using Inversion Mutation, selects a random range within the chromosome and reverses the order of its elements.

In [9]:
def Mutate(c, n, rng):

    # Choose random chunk 
    rev_range = rng.choice(n, size=2, replace=False)
    # Put the smaller one first
    rev_range.sort()
    start, end = rev_range[0], rev_range[1]
    
    # Reverse the order of elements
    c[start : end + 1] = c[start : end + 1][::-1]
    

# The Main Evolutionary Loop


In [ ]:
def Evolve(population, n, min_reps, tournament_size, mutation_chance = 0.05, patience= 5, rng= None):
    if rng is None:
        rng = np.random.default_rng()

    pop_size = population.shape[0] # Populatio size   
    fittests = [] # The fittest chromosome from each generation 
    best_f = -1.0 # Keeps the fitness of fittest chromosome to know when to stop after minimum repeats 

    gen_i = 0 # counter
    improving = patience # how many times to continue of result is not getting better
    while gen_i < min_reps or improving:

        # Fitness of every chromosome in the population 
        fitness = np.array([fitness_of(c) for c in population])

        # Find the fittest of current generation 
        fittest_idx = np.argmax(fitness)
        fittests.append(population[fittest_idx])

        # Update the best answer and stop counter
        if fitness[fittest_idx] > best_f :
            best_f = fitness[fittest_idx]
            improving = patience
        else :
            improving -= 1
            
        # Create the next generation same size as current one
        next_generation = []
        while len(next_generation) < pop_size: 
            
            # -- First parent --

            # chooses random chromosomes
            competitioners = rng.choice(population, size=tournament_size, replace=False)
            f = np.array([fitness_of(c) for c in competitioners]) # Calculate their fitness

            # Choose the winner as parent
            parent_a = competitioners[np.argmax(f)]

            # -- Second parent --
            
            parent_b = parent_a
            tries = 0 
            # loop to prevent choosing a single chromosome as both parents 
            while np.array_equal(parent_a, parent_b) and tries < 5:
                # chooses random chromosomes
                competitioners = rng.choice(population, size=tournament_size, replace=False)
                f = np.array([fitness_of(c) for c in competitioners]) # Calculate their fitness

                # Choose the winner as parent
                parent_b = competitioners[np.argmax(f)]
                
                tries += 1

            # -- Crossover --

            child_a = crossover(parent_a, parent_b, n, rng)
            child_b = crossover(parent_b, parent_a, n, rng)

            # -- Mutation --

            if (rng.random() <= mutation_chance):
                Mutate(child_a, n, rng)
            next_generation.append(child_a)

            if (rng.random() <= mutation_chance):
                Mutate(child_b, n, rng)
            next_generation.append(child_a)

        gen_i += 1
        population = np.array(next_generation)

    return fittests

In [24]:
print(Evolve(population,n,100,3,patience=0))


356.95801073720116 [1 3 2 5 9 6 4 7 8 0]
[[5 6 0 7 3 2 4 9 1 8]
 [4 8 2 6 5 9 7 3 0 1]
 [0 9 8 4 7 2 3 5 6 1]
 [6 1 2 7 9 5 8 4 0 3]
 [3 2 6 5 1 4 8 9 0 7]
 [0 5 8 1 2 7 9 6 4 3]
 [3 7 4 9 1 8 6 2 0 5]
 [0 6 5 3 4 8 9 2 1 7]
 [7 6 3 8 1 0 5 2 9 4]
 [0 2 4 6 9 1 3 8 7 5]
 [6 0 2 8 3 4 1 9 5 7]
 [0 1 8 5 2 3 4 9 7 6]
 [5 8 9 1 7 0 4 6 2 3]
 [2 1 4 8 7 6 5 3 0 9]
 [4 0 8 5 7 6 2 9 1 3]
 [9 4 6 0 3 1 7 2 8 5]
 [6 3 7 5 9 0 8 1 2 4]
 [6 8 4 5 9 7 3 1 2 0]
 [8 1 6 7 3 9 4 0 2 5]
 [5 2 7 6 4 9 1 8 3 0]
 [0 6 8 3 4 9 1 2 5 7]
 [8 7 1 4 6 5 0 3 2 9]
 [0 3 8 1 6 9 5 2 4 7]
 [9 6 3 5 2 7 4 0 1 8]
 [6 5 7 8 2 4 9 3 1 0]
 [9 4 1 0 2 5 3 6 8 7]
 [7 3 5 8 4 2 6 0 1 9]
 [7 0 9 4 6 3 8 2 1 5]
 [2 6 9 1 0 7 4 3 5 8]
 [2 3 7 8 5 1 4 0 6 9]
 [3 6 9 0 4 7 5 8 1 2]
 [0 9 8 3 7 4 2 5 6 1]
 [7 1 4 3 8 0 6 5 2 9]
 [5 6 0 7 8 3 2 9 1 4]
 [3 4 8 0 1 6 7 2 5 9]
 [5 4 0 9 3 2 6 8 7 1]
 [7 2 6 8 5 3 9 0 1 4]
 [8 5 7 6 9 3 2 0 1 4]
 [0 1 5 7 8 2 3 9 4 6]
 [4 0 5 2 9 3 7 6 1 8]
 [4 8 7 6 0 1 3 2 9 5]
 [7 5 0 4 3 8 9 

KeyboardInterrupt: 